# ML Assignment 2 — Dry Bean Variety Classification

**M.Tech (AIML), BITS WILP — Machine Learning Assignment 2**

Five classifiers are trained on the UCI Dry Bean dataset and evaluated using Accuracy, AUC, Precision, Recall, F1 and MCC. This notebook includes data checks, EDA, plots, model tuning and hold-out evaluation.

In [ ]:
import json, platform, socket, time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from bean_metrics import BEAN_FEATURES, TARGET_COL, MODEL_FILES, METRIC_NAMES, SEED, score_classifier
from train_models import candidate_models

print("Host:", socket.gethostname(), "| Python", platform.python_version(), "| scikit-learn", sklearn.__version__)
print("Run at:", time.strftime("%Y-%m-%d %H:%M:%S"))


## 1. Load the dataset

In [ ]:
DATA_PATH = Path("data/dry_bean_full.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Dataset not found. Run `python download_data.py` first.")

beans_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", beans_raw.shape)
print("Missing values:", int(beans_raw.isna().sum().sum()),
      "| exact duplicate rows:", int(beans_raw.duplicated().sum()))
beans = beans_raw.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", beans.shape)
display(beans.head())
display(beans[BEAN_FEATURES].describe().T.round(4))


## 2. Exploratory analysis

In [ ]:
counts = beans[TARGET_COL].value_counts()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(counts.index[::-1], counts.values[::-1], color="#2a78d6", height=0.6)
for i, v in enumerate(counts.values[::-1]):
    ax.text(v + 30, i, f"{v:,}", va="center", fontsize=9)
ax.set_title("Samples per bean variety (after de-duplication)")
ax.set_xlabel("Number of samples")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()
print((counts / counts.sum()).round(3).to_dict())


In [ ]:
diverging = sns.diverging_palette(240, 15, as_cmap=True)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(beans[BEAN_FEATURES].corr(), cmap=diverging, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.5, linecolor="white", ax=ax)
ax.set_title("Pearson correlation between the 16 shape features")
plt.tight_layout()
plt.show()


Many size features are strongly correlated (for example, Area, Perimeter, ConvexArea and EquivDiameter). This indicates multicollinearity for linear models and challenges the conditional-independence assumption of Naive Bayes.

## 3. Train / test split

An 80/20 stratified split preserves the class proportions. The test split is saved as `outputs/test_data.csv`.

In [ ]:
X, y = beans[BEAN_FEATURES], beans[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
Path("outputs").mkdir(exist_ok=True)
Path("model").mkdir(exist_ok=True)
pd.concat([X_train, y_train], axis=1).to_csv("outputs/train_data.csv", index=False)
pd.concat([X_test, y_test], axis=1).to_csv("outputs/test_data.csv", index=False)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 4. Train the five models

Each model uses a pipeline with model-appropriate preprocessing. Hyperparameters are selected by 5-fold stratified cross-validation using macro-F1. Only the training split is used for tuning.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fitted, results, tuning = {}, {}, []
for name, (pipe, grid) in candidate_models().items():
    t0 = time.time()
    search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=folds, n_jobs=-1)
    search.fit(X_train, y_train)
    model = search.best_estimator_
    fitted[name] = model
    results[name] = score_classifier(
        y_test, model.predict(X_test), model.predict_proba(X_test), model.classes_
    )
    joblib.dump(model, Path("model") / MODEL_FILES[name], compress=3)
    tuning.append({
        "Model": name, "Best params": search.best_params_,
        "CV macro-F1": round(search.best_score_, 4),
        "Time (s)": round(time.time() - t0, 1)
    })
tuning_df = pd.DataFrame(tuning)
display(tuning_df)


## 5. Evaluation metrics on the hold-out test set

In [ ]:
comparison = pd.DataFrame(results).T[METRIC_NAMES].round(4)
comparison.index.name = "ML Model Name"
display(comparison.style.highlight_max(axis=0, color="#cde2fb"))
print(comparison.to_markdown())


## 6. Confusion matrices and classification reports

In [ ]:
for name, model in fitted.items():
    pred = model.predict(X_test)
    print("\\n" + "=" * 80 + "\\n" + name)
    print(classification_report(y_test, pred, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=model.classes_)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=model.classes_, yticklabels=model.classes_, ax=ax)
    ax.set_title(f"Confusion matrix — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 7. Summary

Compare the models using macro-F1 and MCC in addition to accuracy. Macro averaging gives each bean variety equal weight, which is useful because the class distribution is imbalanced. AUC is calculated as macro one-vs-rest ROC AUC.

In [ ]:
best_model_name = comparison["F1"].idxmax()
print("Best model by macro-F1:", best_model_name)
print(comparison.loc[best_model_name])
